In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

**Dataset**

In [2]:
texts = [
    "I love this movie",
    "This film was amazing",
    "The acting was great",
    "I really enjoyed the story",
    "What a fantastic experience",
    "This was boring",
    "I hate this movie",
    "The plot was terrible",
    "Very bad acting",
    "I did not enjoy this film",
    "The movie was wonderful",
    "Absolutely loved it",
    "It was a great performance",
    "This was a horrible film",
    "Worst movie ever",
    "I liked the soundtrack",
    "The ending was beautiful",
    "The characters were boring",
    "I will watch it again",
    "It was a waste of time"
]

labels = [
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0,
    1, 1, 1, 0, 0,
    1, 1, 0, 1, 0
]

print("Total samples:", len(texts))
print("First sample:", texts[0], labels[0])

Total samples: 20
First sample: I love this movie 1


In [3]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.20, random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))

Train size: 16
Validation size: 4


**Load tokenizer**

In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [5]:
sample_text = train_texts[0]

encoded_sample = tokenizer(
    sample_text,
    padding="max_length",
    truncation=True,
    max_length=16,
    return_tensors="pt"
)

print("Text:", sample_text)
print("Input IDs:", encoded_sample["input_ids"])
print("Attention Mask:", encoded_sample["attention_mask"])
print("Decoded back:", tokenizer.decode(encoded_sample["input_ids"][0]))

Text: Very bad acting
Input IDs: tensor([[ 101, 2200, 2919, 3772,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])
Decoded back: [CLS] very bad acting [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


**Custom Dataset class**

In [8]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=32):
        self.texts = texts
        self.lables = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.lables[idx]
        
        encoding = tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }
        return item

In [9]:
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_length=32)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, max_length=32)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

Train dataset size: 16
Validation dataset size: 4


**Create dataloaders**

In [10]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)

**Inspect one batch**

In [12]:
batch = next(iter(train_loader))

print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)

print("\n\n",batch["input_ids"])
print(batch["attention_mask"])
print(batch["labels"])

input_ids shape: torch.Size([4, 32])
attention_mask shape: torch.Size([4, 32])
labels shape: torch.Size([4])


 tensor([[ 101, 2200, 2919, 3772,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0],
        [ 101, 7078, 3866, 2009,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0],
        [ 101, 5409, 3185, 2412,  102,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0],
        [ 101, 1996, 4566, 2001, 3376,  102,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]])
tensor(

**Load pretrained transformer for classification**

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.to(device)
print(device)
print(model)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


cuda
DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=Fa

**one forward pass**

In [14]:
small_batch = next(iter(train_loader))

input_ids = small_batch["input_ids"].to(device)
attention_mask = small_batch["attention_mask"].to(device)
labels_batch = small_batch["labels"].to(device)

outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    labels=labels_batch
)

print("Loss:", outputs.loss)
print("Logits:", outputs.logits)
print("Logits shape:", outputs.logits.shape)

Loss: tensor(0.6767, device='cuda:0', grad_fn=<NllLossBackward0>)
Logits: tensor([[-0.0596, -0.0589],
        [-0.0424, -0.0722],
        [-0.0014, -0.0462],
        [-0.0134, -0.0711]], device='cuda:0', grad_fn=<AddmmBackward0>)
Logits shape: torch.Size([4, 2])
